# Conversational agent QLoRA fine-tuning

4-bit QLoRA fine-tuning of `unsloth/llama-3-8b-bnb-4bit` on the synthetic scheduling-agent
dataset, using Unsloth's `FastLanguageModel`. This notebook is the interactive twin of
`train.py` in this directory; both read the same `config.yaml`.

**Requires an NVIDIA GPU.** This was developed against a 24 GB L4 on GCP. See
`docs/cost-and-gpu-notes.md` for VRAM behavior and cost. It will not run on a laptop
with no CUDA device. For a GPU-free sanity check of the config and the dataset, run
`train.py --dry-run` instead, from the repo root:

```bash
uv run python recipes/conversational-agent-qlora/train.py --dry-run
```

## 1. Install the training dependencies

These are the `[train]` extra from `pyproject.toml`: torch, unsloth, bitsandbytes,
transformers, trl, accelerate, peft, xformers. Only install this on a Linux machine
with an NVIDIA GPU.

In [ ]:
%pip install -e "..[train]"  # run from recipes/conversational-agent-qlora/, or adjust the path

## 2. Load config.yaml

Same config the script reads, so the notebook and the script never drift apart.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT))

from llm_finetuning_recipes.config import load_config, validate_config
from llm_finetuning_recipes.formatting import format_example, dataset_token_stats
from llm_finetuning_recipes.validation import load_jsonl, validate_dataset, format_report

config = load_config(REPO_ROOT / "recipes/conversational-agent-qlora/config.yaml")
errors = validate_config(config)
assert not errors, errors
config

## 3. Load and validate the dataset

Runs the same checks as `scripts/check_dataset.py`. If this cell reports FAIL, fix the
dataset before spending any GPU time on it.

In [ ]:
train_path = REPO_ROOT / config["data"]["train_path"]
val_path = REPO_ROOT / config["data"]["val_path"]

train_examples = load_jsonl(train_path)
val_examples = load_jsonl(val_path)

print(format_report(validate_dataset(train_examples, path=str(train_path))))
print()
print(format_report(validate_dataset(val_examples, path=str(val_path))))

In [ ]:
stats = dataset_token_stats(train_examples, max_seq_length=config["training"]["max_seq_length"])
print(stats)
print()
print(format_example(train_examples[0]))

## 4. Load the base model in 4-bit with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config["base_model"],
    max_seq_length=config["training"]["max_seq_length"],
    dtype=None,  # auto-detect bf16/fp16 support
    load_in_4bit=True,
)

## 5. Attach the LoRA adapters

`r=16`, `alpha=16`, `dropout=0.1` on the attention and MLP projections. Dropout above 0
disables Unsloth's fused fast-LoRA forward path; this recipe accepts that tradeoff for
the regularization. See `docs/cost-and-gpu-notes.md` and
`docs/lessons-from-five-iterations.md` for why.

In [ ]:
lora_cfg = config["lora"]

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["alpha"],
    lora_dropout=lora_cfg["dropout"],
    target_modules=lora_cfg["target_modules"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=13,
)

## 6. Build the SFT datasets

Each example is formatted into a single Llama-3 chat-template string ending in the
target agent turn, the same formatting `train.py` and the dry run use.

In [ ]:
from datasets import Dataset

def to_text(example):
    return {"text": format_example(example)}

train_dataset = Dataset.from_list(train_examples).map(to_text)
val_dataset = Dataset.from_list(val_examples).map(to_text)
len(train_dataset), len(val_dataset)

## 7. Train

Batch 2, grad accumulation 4 (effective batch 8), 5 epochs, cosine schedule, eval each
epoch at batch size 1. These are the settings that stopped producing NaN losses and
overfit adapters after five iterations; see `docs/lessons-from-five-iterations.md`.

In [ ]:
import torch
from trl import SFTConfig, SFTTrainer

training_cfg = config["training"]

sft_config = SFTConfig(
    output_dir=str(REPO_ROOT / config["output_dir"]),
    per_device_train_batch_size=training_cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=training_cfg["per_device_eval_batch_size"],
    gradient_accumulation_steps=training_cfg["gradient_accumulation_steps"],
    num_train_epochs=training_cfg["num_train_epochs"],
    learning_rate=training_cfg["learning_rate"],
    weight_decay=training_cfg["weight_decay"],
    lr_scheduler_type=training_cfg["lr_scheduler_type"],
    eval_strategy=training_cfg["eval_strategy"],
    max_seq_length=training_cfg["max_seq_length"],
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    args=sft_config,
)

In [ ]:
trainer.train()

## 8. Save the adapter

In [ ]:
trainer.save_model(str(REPO_ROOT / config["output_dir"]))

## 9. Quick manual check

Generate a few turns with the fine-tuned adapter and eyeball them for length and
single-clause shape before running a full `scripts/eval_responses.py` pass.

In [ ]:
FastLanguageModel.for_inference(model)

prompt = format_example({
    "system": train_examples[0]["system"],
    "history": [
        {"speaker": "agent", "text": "Thanks for calling Cedar Creek Home Services, how can I help you today?"},
        {"speaker": "caller", "text": "Hi, I need someone to look at a leaking kitchen faucet."},
    ],
    "target": "",
}).rsplit("<|eot_id|>", 2)[0] + "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=40, do_sample=False)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))